In [ ]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/paper.json').is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
plt.rcdefaults()
out = Path('Figures/Generated')
out.mkdir(parents=True, exist_ok=True)
figure_size = (5.5, 2.2)
height_ratios = [5, 2]
index_size = 5.2
daily_path = Path('FigureSources/daily_plot_data.csv')
sp = pd.read_csv(daily_path, parse_dates=['target_date']).query("market == '^GSPC'").sort_values('target_date')
expected = pd.read_csv('FigureSources/sp500_event_index.csv', parse_dates=['target_date'])
anchor = expected.iloc[0]
anchor_index = np.flatnonzero(sp.target_date.eq(anchor.target_date))[0]
cumulative = sp.actual_log_return.cumsum().to_numpy()
sp['Close'] = anchor.Close * np.exp(cumulative-cumulative[anchor_index])
assert np.allclose(sp.set_index('target_date').loc[expected.target_date,'Close'],expected.Close,rtol=1e-10)
events = sp[sp.actual_label.eq(1)].copy()
events['index'] = np.arange(1, len(events) + 1)
expected = pd.read_csv(Path('FigureSources/sp500_event_index.csv'), parse_dates=['target_date'])
assert events.target_date.tolist() == expected.target_date.tolist() and len(events) == 17 and (len(sp) == 374)
fig, axes = plt.subplots(2, 1, figsize=figure_size, sharex=True, gridspec_kw={'height_ratios': height_ratios})
fig.subplots_adjust(left=0.105, right=0.985, top=0.97, bottom=0.2, hspace=0.12)
x = mdates.date2num(sp.target_date)
axes[0].plot(x, sp.Close, color='#4169E1', lw=1.0, zorder=3)
axes[0].scatter(mdates.date2num(events.target_date), events.Close, color='#B43C4B', s=12, edgecolor='white', linewidth=0.4, zorder=4)
price_range = sp.Close.max() - sp.Close.min()
axes[0].set_ylim(sp.Close.min() - 0.12 * price_range, sp.Close.max() + 0.1 * price_range)
axes[0].set_ylabel('S&P 500', fontsize=8)
returns = sp.actual_log_return * 100
axes[1].bar(x, returns, color=np.where(sp.actual_label.eq(1), '#B43C4B', '#89929D'), width=0.9, linewidth=0, zorder=3)
axes[1].plot(x, sp.crash_threshold * 100, color='#111111', ls=(0, (4, 2)), lw=0.8, zorder=4)
axes[1].axhline(0, color='#949ba4', lw=0.5)
axes[1].set_ylim(returns.min() - 0.4, returns.max() + 0.4)
axes[1].set_ylabel('Log return (%)', fontsize=7)
for ax in axes:
    ax.tick_params(axis='x', length=2, labelsize=7)
    ax.tick_params(axis='y', length=2, labelsize=7)
    ax.yaxis.set_major_locator(MaxNLocator(3))
ticks = pd.date_range('2025-04-01', '2026-08-01', freq='MS')
axes[1].set_xticks(mdates.date2num(ticks), [d.strftime('%y-%m') if d.month == 1 else d.strftime('%m') for d in ticks])
axes[1].set_xlim(x[0] - 4, x[-1] + 4)
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
occupied, annotation_checks = ([], [])
for row in events.itertuples():
    label = axes[0].annotate(str(row.index), (mdates.date2num(row.target_date), row.Close), xytext=(3, 4), textcoords='offset points', fontsize=index_size, fontweight='bold', color='#7A2837', zorder=5)
    for dx, dy in [(3, 4), (3, -4), (-3, 4), (-3, -4), (0, 8), (0, -8), (6, 8), (-6, -8), (-7, 9), (7, -9)]:
        label.set_position((dx, dy))
        label.set_ha('left' if dx > 0 else 'right' if dx < 0 else 'center')
        label.set_va('bottom' if dy >= 0 else 'top')
        box = label.get_window_extent(renderer).expanded(1.08, 1.08)
        if axes[0].bbox.contains(*box.get_points()[0]) and axes[0].bbox.contains(*box.get_points()[1]) and (not any((box.overlaps(previous) for previous in occupied))):
            break
    else:
        raise AssertionError(('label placement', row.index))
    occupied.append(box)
    annotation_checks.append({'index': int(row.index), 'offset_points': [dx, dy]})
assert len(fig.lines) == 0
assert axes[1].get_ylim()[0] < returns.min() and axes[1].get_ylim()[1] > returns.max()
for ax in axes:
    visible_y_labels = [label for value, label in zip(ax.get_yticks(), ax.get_yticklabels()) if min(ax.get_ylim()) <= value <= max(ax.get_ylim())]
    for label in [ax.yaxis.label, *visible_y_labels, *ax.get_xticklabels()]:
        if label.get_visible():
            box = label.get_window_extent(renderer)
            assert fig.bbox.contains(*box.get_points()[0]) and fig.bbox.contains(*box.get_points()[1]), label.get_text()
assert np.isclose(axes[0].get_position().height / axes[1].get_position().height, 2.5)
x_boxes = [label.get_window_extent(renderer) for label in axes[1].get_xticklabels()]
assert not any((left.overlaps(right) for left, right in zip(x_boxes, x_boxes[1:])))
fig.savefig(out / 'Figure A1 Anomaly Events.png', dpi=360, facecolor='white')
